In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [ ]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from gate import IOStore
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
ios   = IOStore()
mdbpd = MusicDBPermDir()

# DB-Specific

In [ ]:
from lib import metalarchives
mio   = metalarchives.MusicDBIO(verbose=False, mkDirs=False)
apiio = metalarchives.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

# Master Files

In [ ]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists       = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [ ]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Master Search:       {0}".format(len(localArtists.get())))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

# Search For New Artists

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

In [ ]:
useStarterData  = False
useMasterData   = False
useRYMData      = True
useSpotifyData  = False
useAllMusicData = False # Last one done
useLastFMData   = False
useAOTYData     = False

if useStarterData is True:
    starterData = io.get("starter.p")
    artistNames = Series({v["Ref"].split("/")[-1]: v["Name"] for k,v in starterData.items()})
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useSpotifyData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    spotio = mdbgate.getIO(db="Spotify")
    spotGenreData = DataFrame(spotio.data.getSummaryNameData()).join(spotio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    spotGenreData = spotGenreData[spotGenreData["Genre"].notna()]
    artistNames = spotGenreData[spotGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useAllMusicData is True:
    from gate import MusicDBGate
    mdbgate = MusicDBGate()
    amio = mdbgate.getIO(db="AllMusic")
    amGenreData = DataFrame(amio.data.getSummaryNameData()).join(amio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom", "Hard"]
            for test in tests:
                for genre in genres:
                    if genre.title().find(test) != -1:
                        return True
            return False

    amGenreData = amGenreData[amGenreData["Tag"].notna()]
    artistNames = amGenreData[amGenreData["Tag"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]   

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet))) 
elif useRYMData is True:
    rymio = ios.get(db="RateYourMusic")
    rymGenreData = DataFrame(rymio.data.getSummaryNameData()).join(rymio.data.getSummaryGenreData())
    def isMetal(genres):
        if isinstance(genres,list):
            tests = ["Metal", "Black", "Death", "Thrash", "Doom"]
            for test in tests:
                for genre in genres:
                    if genre.find(test) != -1:
                        return True
            return False

    rymGenreData = rymGenreData[rymGenreData["Genre"].notna()]
    artistNames = rymGenreData[rymGenreData["Genre"].apply(isMetal)]["Name"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
elif useMasterData is True:
    from musicdb import MusicDBIO
    pdbio = MusicDBIO()
    mmeDF = pdbio.getData()
    artistNames       = mmeDF[mmeDF["AllMusic"].isna()]["ArtistName"]
    masterArtistsDict = masterArtists.get()
    artistNamesToGet  = artistNames[~artistNames.isin(masterArtistsDict.keys())]

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(masterArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    

#   Artist Names To Get:  8443
#   Artist Names To Get:  4486

## Download Artist Searches

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
masterArtistsDict     = masterArtists.get()
masterArtistsDataDict = masterArtistsData.get()
searchedForErrors     = errors.get()
for i,(idx,artistName) in enumerate(artistNamesToGet.iteritems()):
    if masterArtistsDict.get(artistName) is not None:
        continue

    response = apiio.getArtistSearchResults(artistName=artistName)
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistName] = True
        apiio.sleep(3.5)
    
    masterArtistsDict[artistName]     = True
    masterArtistsDataDict[artistName] = response
    apiio.sleep(3.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
        masterArtists.save(data=masterArtistsDict)
        masterArtistsData.save(data=masterArtistsDataDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

if True:
    ts.stop()
    print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(masterArtistsDict), len(masterArtistsDataDict), db))
    masterArtists.save(data=masterArtistsDict)
    masterArtistsData.save(data=masterArtistsDataDict)
    if len(searchedForErrors) > 0:
        errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import concat
df = masterArtistsData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = metalarchives.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF, Series(df)])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    artists = {}
    for artistName,artistResults in searchDF.iteritems():
        if artistResults is not None:
            for item in artistResults:
                artists[item['id']] = item['name']
    print("Found {0} Unique Artists".format(len(artists)))
    s = Series(artists)
    print("  ==> {0} Old Artists".format(len(s[s.index.isin(knownArtists.index)])))
    print("  ==> {0} New Artists".format(len(s[~s.index.isin(knownArtists.index)])))
    print("Saving Data")
    metalarchives.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

In [ ]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = metalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = metalarchives.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())]
artistIDsToGet   = artistIDsToGet[~artistIDsToGet.index.isin(errors.get().keys())]

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

#   Artist IDs To Get:  4619

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "10:50am")
tt = TermTime("today", "9:00pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(5.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
del searchedForErrors['118073']
errors.save(data=searchedForErrors)


In [ ]:
metalarchives.moveLocalFiles()